# Part 1.2 - Analysis of the Provided CT Volume

Loading and inspecting the sample CT volume in `ct_sample/DICOM/`.

In [1]:
import pydicom
import numpy as np
from pathlib import Path

DICOM_DIR = Path("../ct_sample/DICOM")
dcm_files = sorted(DICOM_DIR.glob("*.dcm"))
print(f"Found {len(dcm_files)} DICOM files")

Found 321 DICOM files


Load every slice, sort them into correct anatomical order using `ImagePositionPatient` (the z-coordinate), and stack them into a single 3D volume. Filename order is not trusted since DICOM does not guarantee filenames sort correctly.

In [2]:
slices = [pydicom.dcmread(f) for f in dcm_files]

# sort by z-position (third component of ImagePositionPatient), not filename
slices.sort(key=lambda s: float(s.ImagePositionPatient[2]))

def to_hu(ds):
    slope = float(getattr(ds, "RescaleSlope", 1.0))
    intercept = float(getattr(ds, "RescaleIntercept", 0.0))
    return ds.pixel_array.astype(np.float64) * slope + intercept

volume = np.stack([to_hu(s) for s in slices], axis=0)
volume = volume.astype(np.int16)

print(f"Volume shape: {volume.shape}")
print(f"dtype: {volume.dtype}")
print(f"Slice spacing (mm): {abs(slices[1].ImagePositionPatient[2] - slices[0].ImagePositionPatient[2]):.3f}")
print(f"Pixel spacing (mm): {slices[0].PixelSpacing}")

Volume shape: (321, 512, 512)
dtype: int16
Slice spacing (mm): 0.500
Pixel spacing (mm): [0.43, 0.43]


Report the minimum and maximum intensity values in the volume. CT intensities are expressed in Hounsfield Units (HU), a standardized scale where 0 HU is defined as water and -1000 HU is defined as air. Typical reference ranges: air around -1000, fat around -100 to -50, soft tissue/muscle around 10 to 40, bone from roughly +300 up to +1500 or more for dense cortical bone, and metal implants (the fixation plates/screws) can exceed +3000, often reaching the top of the scanner's representable range.

In [3]:
vol_min = int(volume.min())
vol_max = int(volume.max())

print(f"Minimum intensity: {vol_min} HU")
print(f"Maximum intensity: {vol_max} HU")

Minimum intensity: -16040 HU
Maximum intensity: 32767 HU


Export a small number of representative axial slices as JPG images. Because the raw HU range is dominated by the padding/saturation values found above, the slices are first windowed (clipped to a fixed HU range and rescaled to 0-255) before saving, otherwise the JPGs would look almost entirely black or white. A bone window (center 400 HU, width 1800 HU) is used since bone and the metal fixation hardware are the structures of interest for this project.

In [4]:
from PIL import Image

OUT_DIR = Path("../images/sample_slices")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def window(hu_slice, center=400, width=1800):
    lo, hi = center - width / 2, center + width / 2
    clipped = np.clip(hu_slice, lo, hi)
    scaled = (clipped - lo) / (hi - lo) * 255.0
    return scaled.astype(np.uint8)

n_slices = volume.shape[0]
sample_indices = np.linspace(0, n_slices - 1, 5, dtype=int)

for idx in sample_indices:
    img = window(volume[idx])
    out_path = OUT_DIR / f"slice_{idx:03d}.jpg"
    Image.fromarray(img).save(out_path, quality=95)

print(f"Exported slices: {list(sample_indices)}")
print(f"Saved to: {OUT_DIR.resolve()}")

Exported slices: [np.int64(0), np.int64(80), np.int64(160), np.int64(240), np.int64(320)]
Saved to: E:\Bone Union Detection\images\sample_slices
